# 🚗 Explorador de Recalls Vehiculares (NHTSA API)

**Fuente de datos:** [NHTSA vPIC API](https://vpic.nhtsa.dot.gov/api/) y [NHTSA Recalls API](https://api.nhtsa.gov/) (datos públicos en tiempo real, sin descarga previa).

Este dashboard consulta en vivo la API oficial del **Departamento de Transporte de EE. UU. (NHTSA)** para:
1. Listar los modelos disponibles de una marca/año seleccionados (dropdown dinámico y encadenado).
2. Consultar los **recalls (llamados a revisión)** activos de un modelo específico.
3. Visualizar los recalls por componente afectado y mostrar el detalle en una tabla.

> Ejecuta las celdas en orden. Requiere conexión a internet.

In [1]:
# Importaciones y configuración global
import requests
import pandas as pd
import plotly.express as px
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Forzamos el renderer para que los gráficos se vean embebidos en Jupyter Notebook
pio.renderers.default = "notebook"

NHTSA_VPIC_BASE = "https://vpic.nhtsa.dot.gov/api/vehicles"
NHTSA_RECALLS_BASE = "https://api.nhtsa.gov/recalls/recallsByVehicle"
REQUEST_TIMEOUT = 10  # segundos

# Lista curada de marcas populares (evita cargar las +9,000 marcas del catálogo completo de NHTSA)
MARCAS_DISPONIBLES = [
    "Toyota", "Honda", "Ford", "Chevrolet", "Nissan", "BMW", "Mercedes-Benz",
    "Volkswagen", "Hyundai", "Kia", "Mazda", "Subaru", "Audi", "Jeep", "Tesla",
    "Ram", "GMC", "Volvo", "Lexus", "Mitsubishi"
]

print("Configuración lista. Marcas disponibles:", len(MARCAS_DISPONIBLES))

Configuración lista. Marcas disponibles: 20


In [2]:
# Funciones de consumo de la API (con manejo de errores)

def obtener_modelos(marca: str, anio: int):
    """Consulta NHTSA vPIC y regresa la lista de modelos para una marca y año dados."""
    url = f"{NHTSA_VPIC_BASE}/GetModelsForMakeYear/make/{marca}/modelyear/{anio}?format=json"
    try:
        resp = requests.get(url, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
        data = resp.json()
        resultados = data.get("Results", [])
        modelos = sorted({r["Model_Name"] for r in resultados if r.get("Model_Name")})
        return modelos, None
    except requests.exceptions.RequestException as e:
        return [], f"⚠️ Error de conexión con la API de NHTSA (vPIC): {e}"
    except ValueError:
        return [], "⚠️ La API de NHTSA devolvió una respuesta no válida (JSON malformado)."


def obtener_recalls(marca: str, modelo: str, anio: int):
    """Consulta la API de Recalls de NHTSA para una combinación marca/modelo/año."""
    params = {"make": marca, "model": modelo, "modelYear": anio}
    try:
        resp = requests.get(NHTSA_RECALLS_BASE, params=params, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
        data = resp.json()
        resultados = data.get("results", [])
        if not resultados:
            return pd.DataFrame(), "ℹ️ No se encontraron recalls registrados para esta combinación."
        df = pd.DataFrame(resultados)
        return df, None
    except requests.exceptions.RequestException as e:
        return pd.DataFrame(), f"⚠️ Error de conexión con la API de Recalls: {e}"
    except ValueError:
        return pd.DataFrame(), "⚠️ La API de Recalls devolvió una respuesta no válida."

print("Funciones de consulta definidas correctamente.")

Funciones de consulta definidas correctamente.


In [3]:
# Construcción de los widgets interactivos (encadenados: Marca -> Año -> Modelo)

marca_dd = widgets.Dropdown(options=MARCAS_DISPONIBLES, description="Marca:", value="Toyota")
anio_slider = widgets.IntSlider(value=2022, min=2010, max=2025, step=1, description="Año:", continuous_update=False)
modelo_dd = widgets.Dropdown(options=[], description="Modelo:")
btn_consultar = widgets.Button(description="🔍 Consultar Recalls", button_style="primary")
status_label = widgets.HTML(value="<i>Selecciona marca y año para cargar modelos...</i>")
salida = widgets.Output()

def actualizar_modelos(*args):
    """Se dispara cada vez que cambia Marca o Año: repuebla el dropdown de modelos."""
    status_label.value = "<i>⏳ Consultando modelos disponibles...</i>"
    modelos, error = obtener_modelos(marca_dd.value, anio_slider.value)
    if error:
        status_label.value = f"<span style='color:red'>{error}</span>"
        modelo_dd.options = []
    elif not modelos:
        status_label.value = "<span style='color:orange'>⚠️ No hay modelos registrados para esa combinación marca/año.</span>"
        modelo_dd.options = []
    else:
        modelo_dd.options = modelos
        status_label.value = f"<span style='color:green'>✅ {len(modelos)} modelos encontrados para {marca_dd.value} {anio_slider.value}.</span>"

marca_dd.observe(actualizar_modelos, names="value")
anio_slider.observe(actualizar_modelos, names="value")

# Carga inicial
actualizar_modelos()

controles = widgets.VBox([
    widgets.HBox([marca_dd, anio_slider]),
    modelo_dd,
    status_label,
    btn_consultar
])
display(controles)

In [4]:
# Lógica de consulta de recalls + visualización (gráfico de barras + tabla)

def graficar_recalls(df: pd.DataFrame, marca: str, modelo: str, anio: int):
    conteo = df["Component"].value_counts().reset_index()
    conteo.columns = ["Componente", "Número de Recalls"]

    fig = px.bar(
        conteo.sort_values("Número de Recalls", ascending=True),
        x="Número de Recalls", y="Componente", orientation="h",
        title=f"Recalls por Componente — {marca} {modelo} ({anio})",
        color="Número de Recalls", color_continuous_scale="Reds",
        text="Número de Recalls"
    )
    fig.update_layout(height=max(350, 40 * len(conteo)), template="plotly_white",
                       coloraxis_showscale=False, margin=dict(l=10, r=10, t=60, b=10))
    fig.show()

    tabla = df[["Component", "Summary", "ReportReceivedDate"]].rename(
        columns={"Component": "Componente", "Summary": "Resumen", "ReportReceivedDate": "Fecha de reporte"}
    )
    display(HTML(f"<h4>📋 Detalle de recalls ({len(tabla)} registros)</h4>"))
    display(tabla.style.set_properties(**{"text-align": "left"}))


def on_consultar_clicked(b):
    with salida:
        clear_output(wait=True)
        if not modelo_dd.value:
            print("⚠️ Selecciona un modelo antes de consultar.")
            return
        print(f"⏳ Consultando recalls de {marca_dd.value} {modelo_dd.value} ({anio_slider.value})...")
        df, error = obtener_recalls(marca_dd.value, modelo_dd.value, anio_slider.value)
        clear_output(wait=True)
        if error:
            print(error)
            return
        graficar_recalls(df, marca_dd.value, modelo_dd.value, anio_slider.value)

btn_consultar.on_click(on_consultar_clicked)
display(salida)

Output()

---
### 💡 Notas técnicas
- Los dropdowns de **Modelo** se repueblan dinámicamente (`observe`) cada vez que cambian Marca o Año, consultando la API en tiempo real.
- Toda llamada HTTP está envuelta en `try/except` con timeout, cubriendo caídas de servicio, timeouts y respuestas JSON malformadas.
- No se requiere ningún archivo local: los datos siempre son "en vivo" desde NHTSA.